In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!bash /content/drive/MyDrive/V2_drift/texturemodel/code/run/perform_pca.sh

Streaming output truncated to the last 5000 lines.
Processing real_spatshift_across_scale_crosscorrs, size of array before PCA:
(10000, 20)
Running PCA: original size of array is [9000 x 20]
Time elapsed: 0.00582
Retaining 5 components to explain 95 pct var
Processing pRF 923 of 1456
Index into batch for prf 923: 23
Final size of feature matrix for this image set and pRF is:
(10000, 641)
Processing real_spatshift_across_scale_crosscorrs, size of array before PCA:
(10000, 20)
Running PCA: original size of array is [9000 x 20]
Time elapsed: 0.00630
Retaining 5 components to explain 95 pct var
Processing pRF 924 of 1456
Index into batch for prf 924: 24
Final size of feature matrix for this image set and pRF is:
(10000, 641)
Processing real_spatshift_across_scale_crosscorrs, size of array before PCA:
(10000, 20)
Running PCA: original size of array is [9000 x 20]
Time elapsed: 0.01012
Retaining 5 components to explain 95 pct var
Processing pRF 925 of 1456
Index into batch for prf 925: 25
Fi

In [4]:
import h5py
import numpy as np
from collections import Counter

ROOT = (
    "/content/drive/MyDrive/V2_drift/features/pyramid_texture/"
    "PCA_pcaHL_v95_henderson_replication"
)

feat_file = (
    f"{ROOT}/S8_4ori_4sf_pcaHL_concat_v95_grid5.h5py"
)

label_file = (
    f"{ROOT}/S8_4ori_4sf_featurelabels_pcaHL_v95_grid5.npy"
)

print("=" * 100)
print("FINAL HENDERSON PCA CHECK — S08")
print("=" * 100)

# ------------------------------------------------------------
# HDF5
# ------------------------------------------------------------
with h5py.File(feat_file, "r") as f:
    x = f["features"]

    print("\nHDF5")
    print("shape:", x.shape)
    print("dtype:", x.dtype)

    # small sample only
    sample = x[:100, :, [0, 728, 1455]]

    print("sample finite:", np.isfinite(sample).all())
    print("sample NaNs:", np.isnan(sample).sum())
    print("sample Infs:", np.isinf(sample).sum())


# ------------------------------------------------------------
# Feature labels
# ------------------------------------------------------------
d = np.load(label_file, allow_pickle=True).item()

labels = np.asarray(d["feature_column_labels"]).astype(int)
names = np.asarray(d["feature_type_names"])

print("\nFEATURE LABELS")
print("total:", len(labels))

counts = Counter(labels)

for g in sorted(counts):
    print(
        f"{g:2d} | "
        f"{names[g]:45s} | "
        f"{counts[g]:3d}"
    )

print("\nLL (groups 0–4):", np.sum(labels <= 4))
print("HL (groups 5–13):", np.sum(labels >= 5))
print("TOTAL:", len(labels))

FINAL HENDERSON PCA CHECK — S08

HDF5
shape: (10000, 127, 1456)
dtype: float32
sample finite: False
sample NaNs: 8500
sample Infs: 0

FEATURE LABELS
total: 127
 0 | pixel_stats                                   |   6
 1 | mean_magnitudes                               |  16
 2 | mean_realparts                                |  16
 3 | marginal_stats_lowpass_recons                 |  10
 4 | variance_highpass_resid                       |   1
 5 | magnitude_feature_autocorrs                   |  12
 6 | lowpass_recon_autocorrs                       |   5
 7 | highpass_resid_autocorrs                      |  13
 8 | magnitude_within_scale_crosscorrs             |   6
 9 | real_within_scale_crosscorrs                  |   4
10 | magnitude_across_scale_crosscorrs             |  12
11 | real_imag_across_scale_crosscorrs             |  15
12 | real_spatshift_within_scale_crosscorrs        |   4
13 | real_spatshift_across_scale_crosscorrs        |   7

LL (groups 0–4): 49
HL (groups 5–13): 78


In [6]:
import h5py
import numpy as np

fn = (
    "/content/drive/MyDrive/V2_drift/features/pyramid_texture/"
    "PCA_pcaHL_v95_henderson_replication/"
    "S8_4ori_4sf_pcaHL_concat_v95_grid5.h5py"
)

# Final group boundaries from the feature labels
groups = {
    "magnitude_feature_autocorrs": (49, 61),
    "lowpass_recon_autocorrs": (61, 66),
    "highpass_resid_autocorrs": (66, 79),
    "magnitude_within_scale_crosscorrs": (79, 85),
    "real_within_scale_crosscorrs": (85, 89),
    "magnitude_across_scale_crosscorrs": (89, 101),
    "real_imag_across_scale_crosscorrs": (101, 116),
    "real_spatshift_within_scale_crosscorrs": (116, 120),
    "real_spatshift_across_scale_crosscorrs": (120, 127),
}

with h5py.File(fn, "r") as f:
    x = f["features"]

    for p in [0, 100, 728, 1000, 1455]:

        print("\n" + "=" * 90)
        print("pRF", p)
        print("=" * 90)

        for name, (a, b) in groups.items():

            # one image is enough because entire padded columns
            # appear to be NaN
            vals = x[0, a:b, p]

            valid = np.isfinite(vals)

            print(
                f"{name:45s} "
                f"allocated={b-a:2d}  "
                f"valid={valid.sum():2d}  "
                f"NaN={np.isnan(vals).sum():2d}"
            )


pRF 0
magnitude_feature_autocorrs                   allocated=12  valid=11  NaN= 1
lowpass_recon_autocorrs                       allocated= 5  valid= 2  NaN= 3
highpass_resid_autocorrs                      allocated=13  valid=11  NaN= 2
magnitude_within_scale_crosscorrs             allocated= 6  valid= 6  NaN= 0
real_within_scale_crosscorrs                  allocated= 4  valid= 4  NaN= 0
magnitude_across_scale_crosscorrs             allocated=12  valid=12  NaN= 0
real_imag_across_scale_crosscorrs             allocated=15  valid=15  NaN= 0
real_spatshift_within_scale_crosscorrs        allocated= 4  valid= 4  NaN= 0
real_spatshift_across_scale_crosscorrs        allocated= 7  valid= 7  NaN= 0

pRF 100
magnitude_feature_autocorrs                   allocated=12  valid=11  NaN= 1
lowpass_recon_autocorrs                       allocated= 5  valid= 2  NaN= 3
highpass_resid_autocorrs                      allocated=13  valid=11  NaN= 2
magnitude_within_scale_crosscorrs             allocated= 6  